# 05 Reranker：检索结果重排序

## 学习目标

1. 理解 Bi-Encoder 与 Cross-Encoder
2. 掌握 Cross-Encoder 工作原理
3. 使用 BGE Reranker
4. 实现 Hybrid + Reranker
5. 对比 Reranker 前后的检索效果
6. 掌握 Reranker 在真实 RAG 系统中的应用

In [2]:
import torch
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"

reranker_model = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    max_length=512,
    device=device
)

print("Device:", device)
print("Reranker loaded successfully!")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Device: cuda
Reranker loaded successfully!


In [3]:
query = "持续降雨为什么会增加滑坡风险？"

candidates = [
    "持续降雨会导致土体含水率增加，并可能提高孔隙水压力。",

    "地下水位升高可能降低边坡稳定性。",

    "太阳辐射、云量和温度会影响光伏组件的发电效率。"
]

In [1]:
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

Python: c:\Users\Administrator\miniconda3\envs\agent_project\python.exe
PyTorch: 2.14.1+cu132
PyTorch CUDA: 13.2
CUDA available: True
GPU count: 1
GPU 0: NVIDIA GeForce RTX 3060


In [4]:
pairs = [
    [query, doc]
    for doc in candidates
]

scores = reranker_model.predict(
    pairs,
    batch_size=8,
    show_progress_bar=False
)

for doc, score in sorted(
    zip(candidates, scores),
    key=lambda x: x[1],
    reverse=True
):
    print(f"Score: {float(score):.4f}")
    print(doc)
    print("-" * 60)

Score: 0.7693
持续降雨会导致土体含水率增加，并可能提高孔隙水压力。
------------------------------------------------------------
Score: 0.0123
地下水位升高可能降低边坡稳定性。
------------------------------------------------------------
Score: 0.0000
太阳辐射、云量和温度会影响光伏组件的发电效率。
------------------------------------------------------------


In [5]:
def rerank_documents(
    query,
    documents,
    top_k=3
):
    if not documents:
        return []

    pairs = [
        [query, doc["text"]]
        for doc in documents
    ]

    scores = reranker_model.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False
    )

    reranked = []

    for doc, score in zip(documents, scores):
        item = doc.copy()
        item["rerank_score"] = float(score)
        reranked.append(item)

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [7]:
documents = [
    "持续降雨会增加土体含水率，并可能提高孔隙水压力。",
    "地下水位升高可能降低边坡稳定性。",
    "坡度是影响滑坡发生的重要地形因素。",
    "太阳辐射是影响光伏发电功率的重要因素。",
    "高温可能降低光伏组件的转换效率。",
    "WRF 4.5.1 是一种数值天气预报模型版本。",
    "ERA5-Land 提供陆面气象再分析数据。",
    "RMSE 和 MAE 是常见的预测误差评价指标。",
    "PyTorch 可以用于深度学习模型训练。",
    "BGE-M3 支持多语言文本检索。",
    "Qdrant 是支持向量检索的数据库。",
    "Noah-MP 是一种陆面过程模型。"
]

In [8]:
import re
import jieba
import numpy as np

def tokenize(text):
    # 统一英文字母大小写
    text = text.lower()

    # 保留专业缩写、数字和中文
    tokens = jieba.lcut(text)

    return [
        token.strip()
        for token in tokens
        if token.strip()
        and not re.fullmatch(r"\W+", token)
    ]
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    tokenize(doc)
    for doc in documents
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.474 seconds.
Prefix dict has been built successfully.


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-zh-v1.5"
)
document_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [10]:
def bm25_search(query, top_k=3):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    ranked_ids = np.argsort(-scores)[:top_k]

    return [
        {
            "id": int(idx),
            "text": documents[idx],
            "score": float(scores[idx])
        }
        for idx in ranked_ids
    ]
def dense_search(query, top_k=3):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    scores = document_embeddings @ query_embedding

    ranked_ids = np.argsort(-scores)[:top_k]

    return [
        {
            "id": int(idx),
            "text": documents[idx],
            "score": float(scores[idx])
        }
        for idx in ranked_ids
    ]

# RRF
def reciprocal_rank_fusion(
    result_lists,
    rrf_k=60,
    top_k=3
):
    fused_scores = {}
    document_map = {}

    for results in result_lists:

        for rank, result in enumerate(results, start=1):

            doc_id = result["id"]

            fused_scores[doc_id] = (
                fused_scores.get(doc_id, 0.0)
                + 1.0 / (rrf_k + rank)
            )

            document_map[doc_id] = result["text"]

    ranked_ids = sorted(
        fused_scores,
        key=lambda doc_id: (-fused_scores[doc_id], doc_id)
    )

    return [
        {
            "id": doc_id,
            "text": document_map[doc_id],
            "score": fused_scores[doc_id]
        }
        for doc_id in ranked_ids[:top_k]
    ]

In [13]:
def hybrid_search(
    query,
    top_k=3,
    candidate_k=8,
    rrf_k=60
):
    bm25_results = bm25_search(
        query,
        top_k=candidate_k
    )

    dense_results = dense_search(
        query,
        top_k=candidate_k
    )

    fused_results = reciprocal_rank_fusion(
        result_lists=[
            bm25_results,
            dense_results
        ],
        rrf_k=rrf_k,
        top_k=top_k
    )

    return fused_results

def hybrid_rerank_search(
    query,
    top_k=3,
    candidate_k=8,
    rrf_k=60
):
    # 1. BM25 检索
    bm25_results = bm25_search(
        query,
        top_k=candidate_k
    )

    # 2. Dense 检索
    dense_results = dense_search(
        query,
        top_k=candidate_k
    )

    # 3. RRF 融合
    candidates = reciprocal_rank_fusion(
        result_lists=[
            bm25_results,
            dense_results
        ],
        rrf_k=rrf_k,
        top_k=candidate_k
    )

    # 4. Cross-Encoder 重排序
    final_results = rerank_documents(
        query=query,
        documents=candidates,
        top_k=top_k
    )

    return final_results

In [14]:
test_queries = [
    "持续降雨为什么会增加滑坡风险？",
    "WRF 4.5.1 是什么？",
    "地下水位升高为什么会影响边坡稳定性？"
]

for query in test_queries:

    print("\n" + "=" * 80)
    print("Query:", query)

    before = hybrid_search(
        query,
        top_k=3,
        candidate_k=8
    )

    after = hybrid_rerank_search(
        query,
        top_k=3,
        candidate_k=8
    )

    print("\n[RRF 排序]")
    for rank, item in enumerate(before, 1):
        print(rank, item["text"])

    print("\n[Reranker 排序]")
    for rank, item in enumerate(after, 1):
        print(
            rank,
            round(item["rerank_score"], 4),
            item["text"]
        )


Query: 持续降雨为什么会增加滑坡风险？

[RRF 排序]
1 持续降雨会增加土体含水率，并可能提高孔隙水压力。
2 坡度是影响滑坡发生的重要地形因素。
3 地下水位升高可能降低边坡稳定性。

[Reranker 排序]
1 0.7968 持续降雨会增加土体含水率，并可能提高孔隙水压力。
2 0.0369 坡度是影响滑坡发生的重要地形因素。
3 0.0123 地下水位升高可能降低边坡稳定性。

Query: WRF 4.5.1 是什么？

[RRF 排序]
1 WRF 4.5.1 是一种数值天气预报模型版本。
2 地下水位升高可能降低边坡稳定性。
3 RMSE 和 MAE 是常见的预测误差评价指标。

[Reranker 排序]
1 0.9999 WRF 4.5.1 是一种数值天气预报模型版本。
2 0.001 ERA5-Land 提供陆面气象再分析数据。
3 0.0001 Noah-MP 是一种陆面过程模型。

Query: 地下水位升高为什么会影响边坡稳定性？

[RRF 排序]
1 地下水位升高可能降低边坡稳定性。
2 持续降雨会增加土体含水率，并可能提高孔隙水压力。
3 坡度是影响滑坡发生的重要地形因素。

[Reranker 排序]
1 0.9998 地下水位升高可能降低边坡稳定性。
2 0.0032 坡度是影响滑坡发生的重要地形因素。
3 0.0017 持续降雨会增加土体含水率，并可能提高孔隙水压力。


In [15]:
evaluation_dataset = [
    {
        "query": "持续降雨会不会提高孔隙水压力？",
        "relevant_ids": [0],
        "type": "keyword"
    },
    {
        "query": "连续降水是否容易诱发坡体失稳？",
        "relevant_ids": [0, 1, 2],
        "type": "semantic"
    },
    {
        "query": "地下水位升高会影响边坡吗？",
        "relevant_ids": [1],
        "type": "semantic"
    },
    {
        "query": "影响太阳能发电的气象因素有哪些？",
        "relevant_ids": [3, 4],
        "type": "semantic"
    },
    {
        "query": "WRF 4.5.1 是什么？",
        "relevant_ids": [5],
        "type": "exact"
    },
    {
        "query": "ERA5-Land 是什么数据？",
        "relevant_ids": [6],
        "type": "exact"
    },
    {
        "query": "RMSE 和 MAE 用来评价什么？",
        "relevant_ids": [7],
        "type": "exact"
    },
    {
        "query": "Noah-MP 是什么模型？",
        "relevant_ids": [11],
        "type": "exact"
    }
]

def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    top_k_ids = set(retrieved_ids[:k])
    relevant_ids = set(relevant_ids)

    return int(bool(top_k_ids & relevant_ids))


def recall_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    if not relevant_ids:
        return 0.0

    top_k_ids = set(retrieved_ids[:k])

    return len(top_k_ids & relevant_ids) / len(relevant_ids)


def reciprocal_rank_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    for rank, doc_id in enumerate(retrieved_ids[:k], 1):
        if doc_id in relevant_ids:
            return 1.0 / rank

    return 0.0

# 统一评价函数：
import pandas as pd
import time

def evaluate_search(
    dataset,
    search_func,
    method_name,
    top_k=3
):
    records = []

    for item in dataset:
        query = item["query"]
        relevant_ids = item["relevant_ids"]

        start = time.perf_counter()

        results = search_func(
            query,
            top_k=top_k
        )

        latency_ms = (
            time.perf_counter() - start
        ) * 1000

        retrieved_ids = [
            result["id"]
            for result in results
        ]

        records.append({
            "method": method_name,
            "type": item["type"],
            "query": query,
            "relevant_ids": relevant_ids,
            "retrieved_ids": retrieved_ids,
            "Hit@3": hit_rate_at_k(
                retrieved_ids, relevant_ids, top_k
            ),
            "Recall@3": recall_at_k(
                retrieved_ids, relevant_ids, top_k
            ),
            "RR@3": reciprocal_rank_at_k(
                retrieved_ids, relevant_ids, top_k
            ),
            "latency_ms": latency_ms
        })

    return pd.DataFrame(records)

In [16]:
# 四种检索方法统一评估

methods = {
    "BM25": bm25_search,
    "Dense": dense_search,
    "Hybrid-RRF": hybrid_search,
    "Hybrid-Reranker": hybrid_rerank_search
}

all_eval_results = []

for name, search_func in methods.items():

    print(f"Evaluating: {name}")

    result = evaluate_search(
        dataset=evaluation_dataset,
        search_func=search_func,
        method_name=name,
        top_k=3
    )

    all_eval_results.append(result)

comparison_df = pd.concat(
    all_eval_results,
    ignore_index=True
)

summary_df = (
    comparison_df
    .groupby("method")[
        ["Hit@3", "Recall@3", "RR@3", "latency_ms"]
    ]
    .mean()
    .reset_index()
    .rename(columns={"RR@3": "MRR@3"})
)

display(summary_df.round(4))

Evaluating: BM25
Evaluating: Dense
Evaluating: Hybrid-RRF
Evaluating: Hybrid-Reranker


,method,Hit@3,Recall@3,MRR@3,latency_ms
0,BM25,1.0,0.9375,1.0,0.1914
1,Dense,1.0,1.0000,1.0,6.4503
2,Hybrid-RRF,1.0,1.0000,1.0,5.4692
3,Hybrid-Reranker,1.0,1.0000,1.0,40.2810


In [17]:
import json
import os

os.makedirs("evaluation_results", exist_ok=True)

comparison_df.to_json(
    "evaluation_results/reranker_details.json",
    orient="records",
    force_ascii=False,
    indent=4
)

summary_df.to_csv(
    "evaluation_results/reranker_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation results saved!")

Evaluation results saved!
